**Human Movement Intelligence | Presented by: Reza Saadatyar (2026)**

**08 | Out-of-Fold Confidence and Subject-Disjoint Temperature Calibration**

![HARTH confidence and temperature calibration workflow](../../images/08_consolidated_workflow.svg)

**Objective:** Characterise uncalibrated softmax confidence, class errors and reliability in participant-disjoint Development folds; then separately evaluate temperature scaling fitted on Calibration-only participants.

**Data:** Official NTNU AI Lab HARTH v2.0, eight documented activity classes, six accelerometer channels (50 Hz), 29 Development participants and nested base/expanded temporal windows. Participants S023 and S032 remain excluded.

**Experiment A:** Train one weighted temporal CNN per outer fold for 400 optimiser steps on the expanded windows of Training participants. Evaluate uncalibrated softmax probabilities on held-out base windows. Inspect accuracy, ECE, Brier, class support and confident mistakes.

**Experiment B:** Fit separate weighted CNNs, withholding Calibration participants from model Training inside each outer fold. Fit a positive scalar temperature using Calibration-only labels and evaluate raw versus scaled probabilities on the untouched outer Validation participants.

**Boundary:** The two experiments share source data and outer folds but use distinct model fits and training populations. Temperature scaling preserves top-1 predictions, and improvement in one probability metric does not guarantee improvement in others. Development-only results will be interpreted after running this integrated notebook.


**00. Colab Repository Setup**


In [ ]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess

repo_url = "https://github.com/RezaSaadatyar/human-movement-intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")

if not repo_path.exists():
    subprocess.run(["git", "clone", "-q", repo_url, str(repo_path)], check=True)

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}


**01. Imports and Research Environment**

Load reusable HARTH, fixed-step CNN, confidence-audit, calibration and visualisation utilities. Both experiments write current evidence into the study output directory.


In [ ]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("wearable_activity_recognition")

# ----------------- 2. Scientific Computing and Data Analysis -----------------
import numpy as np
import pandas as pd

# -------------------- 3. Machine Learning and Evaluation ---------------------
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.metrics import f1_score

# ------------------- 4. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure

plot_style(
    font_family="DejaVu Sans", title_size=12.5, title_weight="normal",
    label_size=11.5, label_weight="normal", tick_size=11, legend_size=10,
    figure_dpi=100, figure_size=(9, 4),
)

# ---------------------- 5. HARTH and Research Utilities ----------------------
from harth_data import (
    HARTH_SUBJECTS, ACTIVITY_NAMES, SENSOR_COLUMNS,
    prepare_harth_nested_window_cohort,
)
from core.temporal_cnn import (
    train_temporal_cnn_fixed_steps, predict_temporal_cnn_probabilities,
)
from core.prediction_reliability import audit_prediction_confidence
from core.temperature_calibration import (
    fit_temperature, apply_temperature, multiclass_nll,
)
from core.output_download import download_outputs_from_colab

# ------------------------ 6. Device and Output Folder ------------------------
from core.computer_device import computer_device
device = computer_device()
DEVICE = device.type
output_dir = root / "wearable_activity_recognition" / "output"
output_dir.mkdir(parents=True, exist_ok=True)
print("Research output directory:", output_dir)


**02. Prepare Shared Nested Development Windows**

Load the label-pure, timestamp-continuous HARTH windows once and create one source-aligned set of base held-out rows and outer participant folds. The subsequent confidence and calibration experiments fit separate models on this shared source.


In [ ]:
# =============== 02. Prepare Shared Nested Development Windows ===============

# --------------------- 1. Define Development Population ----------------------
RESERVED_TEST = {"S023", "S032"}
DEVELOPMENT_SUBJECTS = sorted(set(HARTH_SUBJECTS) - RESERVED_TEST)
TASK_CLASSES = tuple(range(1, 9))

# ------------------- 2. Construct Strictly Nested Windows --------------------
X, metadata, overview = prepare_harth_nested_window_cohort(
    subjects=DEVELOPMENT_SUBJECTS,
    data_dir=root / "data/raw/harth_v2",
    task_classes=TASK_CLASSES,
    base_cap=500, expanded_cap=1000,
    window_size=100, step_size=50, sampling_rate=50,
)
base_rows = np.flatnonzero(metadata["in_base_500"].to_numpy(dtype=bool))
y = metadata["label"].to_numpy()
group_ids = metadata["subject_id"].to_numpy()

# ---------------------- 3. Protect Source-Row Alignment ----------------------
if len(X) != len(metadata) or X.shape[1:] != (len(SENSOR_COLUMNS), 100):
    raise ValueError("HARTH source windows and metadata are misaligned.")
if not len(base_rows) or set(group_ids) & RESERVED_TEST:
    raise ValueError("The Development population is empty or includes Test.")
if not np.isin(y, TASK_CLASSES).all():
    raise ValueError("A retained window has an undefined activity label.")

# --------------------- 4. Configure Matching Outer Folds ---------------------
FOLDS, STEPS = 3, 400
outer_folds = list(GroupKFold(n_splits=FOLDS).split(
    np.zeros(len(base_rows)), y[base_rows], groups=group_ids[base_rows],
))
print("Development people:", len(set(group_ids)))
print("Base and expanded source windows:", len(base_rows), len(X))
display(overview.head())


**03. Generate Uncalibrated Out-of-Fold CNN Probabilities**

Fit three independent weighted CNNs using expanded windows belonging to outer Training participants only. Generate one uncalibrated softmax vector per held-out base window; preserve its original source row, class-column order and participant identity.


In [ ]:
# ========== 03. Generate Uncalibrated Out-of-Fold CNN Probabilities ==========

# ------------------------- 1. Fit Three Outer Models -------------------------
confidence_records = []
confidence_fold_metrics = []

for fold, (train, valid) in enumerate(outer_folds, start=1):
    base_train = base_rows[train]
    validation = base_rows[valid]
    train_subjects = set(group_ids[base_train])
    validation_subjects = set(group_ids[validation])
    if train_subjects & validation_subjects:
        raise ValueError("Training and held-out participants overlap.")
    train_rows = np.flatnonzero(np.isin(group_ids, list(train_subjects)))

    model, history, predicted, train_stats = train_temporal_cnn_fixed_steps(
        X_train=X[train_rows], y_train=y[train_rows],
        X_validation=X[validation], y_validation=y[validation],
        classes=TASK_CLASSES, max_steps=STEPS, eval_every=100,
        batch_size=128, learning_rate=1e-3, seed=42,
        device=DEVICE, class_weighting="inverse_sqrt",
    )
    probabilities = predict_temporal_cnn_probabilities(
        model, X[validation], train_stats, TASK_CLASSES,
        batch_size=256, device=DEVICE,
    )
    if not np.array_equal(
        np.asarray(TASK_CLASSES)[probabilities.argmax(axis=1)], predicted
    ):
        raise ValueError("Softmax and final-step predictions disagree.")
    confidence_records.append(pd.DataFrame({
        "fold": fold, "source_row": validation,
        "subject_id": group_ids[validation],
        "true_label": y[validation],
        **{f"prob_{code}": probabilities[:, j]
           for j, code in enumerate(TASK_CLASSES)},
    }))
    confidence_fold_metrics.append({
        "fold": fold,
        "validation_subjects": len(validation_subjects),
        "validation_windows": len(validation),
        "macro_f1": f1_score(
            y[validation], predicted, labels=TASK_CLASSES,
            average="macro", zero_division=0,
        ),
    })
    print("Completed uncalibrated confidence fold:", fold)

# ---------------------- 2. Combine Held-Out Source Rows ----------------------
confidence_oof = pd.concat(
    confidence_records, ignore_index=True
).sort_values("source_row").reset_index(drop=True)
display(pd.DataFrame(confidence_fold_metrics).round(4))


**04. Verify Source Alignment and Audit Softmax Reliability**

Require one held-out prediction per base source window and preserve its exact original labels. Compute fixed-bin ECE, multiclass Brier, accuracy and high-confidence incorrect predictions; these are descriptive confidence diagnostics, not validated uncertainty.


In [ ]:
# ========= 04. Verify Source Alignment and Audit Softmax Reliability =========

# ----------------------- 1. Check Held-Out Provenance ------------------------
if len(confidence_oof) != len(base_rows):
    raise ValueError("Uncalibrated out-of-fold row count is incomplete.")
if not confidence_oof["source_row"].is_unique:
    raise ValueError("Duplicate source rows in confidence evidence.")
np.testing.assert_array_equal(
    confidence_oof["source_row"].to_numpy(), np.sort(base_rows),
)
np.testing.assert_array_equal(
    confidence_oof["true_label"].to_numpy(), y[np.sort(base_rows)],
)

# --------------------- 2. Compute Confidence Diagnostics ---------------------
confidence_prob_columns = [f"prob_{code}" for code in TASK_CLASSES]
(
    confidence_details, confidence_reliability,
    confidence_by_subject, confidence_summary,
) = audit_prediction_confidence(
    y_true=confidence_oof["true_label"].to_numpy(),
    probabilities=confidence_oof[confidence_prob_columns].to_numpy(),
    subjects=confidence_oof["subject_id"].to_numpy(),
    classes=TASK_CLASSES, n_bins=10,
)
confidence_details["fold"] = confidence_oof["fold"].to_numpy()
confidence_details["source_row"] = confidence_oof["source_row"].to_numpy()

# ------------------------ 3. Inspect Recorded Metrics ------------------------
display(pd.DataFrame([confidence_summary]).round(4))
display(confidence_reliability.round(4))


**05. Visualise Uncalibrated Reliability and Confidence**

Compare mean softmax confidence and observed correctness within ten fixed-width bins, then inspect confidence distributions for correct and incorrect predictions. Confidence bins and pooled overlapping windows do not provide independent estimates of certainty.


In [ ]:
# =========== 05. Visualise Uncalibrated Reliability and Confidence ===========

# -------------------------- 1. Reliability Diagram ---------------------------
used_bins = confidence_reliability.loc[
    confidence_reliability["windows"] > 0
]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot([0, 1], [0, 1], linestyle="--",
             label="Ideal reliability reference")
axes[0].plot(
    used_bins["mean_confidence"], used_bins["observed_accuracy"],
    marker="o", label="Held-out Development windows",
)
axes[0].set(xlim=(0, 1), ylim=(0, 1),
            xlabel="Mean softmax confidence", ylabel="Observed accuracy",
            title="Out-of-fold reliability")
axes[0].legend()
axes[0].grid(alpha=0.2)

# ------------------ 2. Correct Versus Incorrect Confidence -------------------
correct = confidence_details.loc[
    confidence_details["correct"], "confidence"
]
wrong = confidence_details.loc[
    ~confidence_details["correct"], "confidence"
]
axes[1].hist([correct, wrong], bins=np.linspace(0, 1, 21),
             label=["Correct", "Incorrect"], alpha=0.7)
axes[1].set(xlabel="Maximum softmax probability", ylabel="Windows",
            title="Prediction confidence and correctness")
axes[1].legend()
fig.tight_layout()
save_figure(fig, output_dir / "08_confidence_reliability.png")
plt.show()


**06. Inspect Minority-Class Errors and Confident Mistakes**

Summarise the support, recall, mistakes and high-confidence errors of each activity. Review Shuffling and stair-class errors and identify participants with concentrated confident mistakes. These descriptions do not introduce a new prediction threshold.


In [ ]:
# ========= 06. Inspect Minority-Class Errors and Confident Mistakes ==========

# ----------------------- 1. Class-Level Error Summary ------------------------
class_rows = []
for code in TASK_CLASSES:
    part = confidence_details.loc[
        confidence_details["true_label"] == code
    ]
    mistakes = part.loc[~part["correct"]]
    class_rows.append({
        "class": ACTIVITY_NAMES[code],
        "support": len(part),
        "recall": float(part["correct"].mean()) if len(part) else np.nan,
        "errors": len(mistakes),
        "high_confidence_errors_0_90": int(
            (mistakes["confidence"] >= 0.90).sum()
        ),
        "mean_error_confidence": (
            float(mistakes["confidence"].mean()) if len(mistakes) else np.nan
        ),
    })
confidence_class_report = pd.DataFrame(class_rows)
display(confidence_class_report.round(4))

# ------------------------ 2. Participant-Level Errors ------------------------
display(
    confidence_by_subject.sort_values(
        "high_confidence_errors", ascending=False
    ).round(4)
)

# ----------------------- 3. Confident Confusion Pairs ------------------------
confident_errors = confidence_details.loc[
    (~confidence_details["correct"]) &
    (confidence_details["confidence"] >= 0.90)
]
confidence_confusion_pairs = (
    confident_errors.groupby(["true_label", "predicted_label"])
    .size().rename("windows").reset_index()
    .sort_values("windows", ascending=False)
)
display(confidence_confusion_pairs.head(15))


**07. Export Uncalibrated Confidence Evidence**

Save fold metrics, class-aligned uncalibrated probabilities, detailed source-window diagnostics, reliability bins, per-person metrics and minority-class error summaries as `08_confidence_` CSV/PNG outputs. The calibration experiment is not fitted from these out-of-fold predictions.


In [ ]:
# ================ 07. Export Uncalibrated Confidence Evidence ================

# ----------------------- 1. Summarise Confidence Audit -----------------------
print("Out-of-fold confidence windows:", len(confidence_oof))
print("Contributing Development participants:",
      confidence_details["subject_id"].nunique())
print("High-confidence mistakes:",
      confidence_summary["high_confidence_errors_0_90"])

# ------------------------- 2. Save Confidence Tables -------------------------
overview.to_csv(
    output_dir / "08_development_cohort_summary.csv", index=False
)
pd.DataFrame(confidence_fold_metrics).to_csv(
    output_dir / "08_confidence_fold_metrics.csv", index=False
)
confidence_oof.to_csv(
    output_dir / "08_confidence_oof_class_probabilities.csv", index=False
)
confidence_details.to_csv(
    output_dir / "08_confidence_prediction_details.csv", index=False
)
confidence_reliability.to_csv(
    output_dir / "08_confidence_reliability_bins.csv", index=False
)
confidence_by_subject.to_csv(
    output_dir / "08_confidence_subject_metrics.csv", index=False
)
pd.DataFrame([confidence_summary]).to_csv(
    output_dir / "08_confidence_summary.csv", index=False
)
confidence_class_report.to_csv(
    output_dir / "08_confidence_class_error_summary.csv", index=False
)
confidence_confusion_pairs.to_csv(
    output_dir / "08_confidence_high_confidence_error_pairs.csv", index=False
)


**08. Fit Subject-Disjoint Temperature Scaling**

Fit separate weighted CNNs and positive scalar temperatures in the same outer GroupKFold assignments. Within each outer Training group, reserve approximately 20% of participants for Calibration; use only the remaining participants' expanded windows for CNN training. Fit temperature from base-cap Calibration labels and evaluate uncalibrated/scaled probabilities only on outer Validation participants.


In [ ]:
# =============== 08. Fit Subject-Disjoint Temperature Scaling ================

# ---------------- 1. Prepare Separate Calibration Experiment -----------------
calibration_records = []
calibration_fold_records = []
calibration_subject_assignments = []

# ------------------- 2. Fit Models and Temperature Scalars -------------------
for fold, (outer_train, outer_valid) in enumerate(outer_folds, start=1):
    outer_train_rows = base_rows[outer_train]
    validation_rows = base_rows[outer_valid]
    inner = GroupShuffleSplit(
        n_splits=1, test_size=0.2, random_state=42 + fold,
    )
    inner_train, inner_cal = next(inner.split(
        np.zeros(len(outer_train_rows)),
        y[outer_train_rows], groups=group_ids[outer_train_rows],
    ))
    train_people = set(group_ids[outer_train_rows[inner_train]])
    cal_people = set(group_ids[outer_train_rows[inner_cal]])
    valid_people = set(group_ids[validation_rows])
    if train_people & cal_people or train_people & valid_people:
        raise ValueError("Calibration Train participants are not disjoint.")
    if cal_people & valid_people:
        raise ValueError("Calibration participants overlap outer Validation.")

    for role, people in (
        ("train", train_people),
        ("calibration", cal_people),
        ("outer_validation", valid_people),
    ):
        calibration_subject_assignments.extend(
            {"fold": fold, "role": role, "subject_id": person}
            for person in sorted(people)
        )

    train_rows = np.flatnonzero(
        np.isin(group_ids, list(train_people))
    )
    cal_rows = outer_train_rows[inner_cal]
    if not set(TASK_CLASSES).issubset(set(y[train_rows])):
        raise ValueError("The CNN Train partition lacks a target activity.")

    print(
        f"Fold {fold}: Train {len(train_people)}, "
        f"Calibration {len(cal_people)}, "
        f"outer Validation {len(valid_people)} participants"
    )
    model, history, _, stats = train_temporal_cnn_fixed_steps(
        X[train_rows], y[train_rows], X[cal_rows], y[cal_rows],
        classes=TASK_CLASSES, max_steps=STEPS, eval_every=STEPS,
        batch_size=128, learning_rate=1e-3, seed=42,
        device=DEVICE, class_weighting="inverse_sqrt",
    )
    cal_prob = predict_temporal_cnn_probabilities(
        model, X[cal_rows], stats, TASK_CLASSES, device=DEVICE,
    )
    fitted = fit_temperature(cal_prob, y[cal_rows], TASK_CLASSES)
    raw = predict_temporal_cnn_probabilities(
        model, X[validation_rows], stats, TASK_CLASSES, device=DEVICE,
    )
    scaled = apply_temperature(raw, fitted["temperature"])

    for method, probabilities in (
        ("Uncalibrated", raw),
        ("Temperature scaled", scaled),
    ):
        _, _, _, metrics = audit_prediction_confidence(
            y[validation_rows], probabilities,
            group_ids[validation_rows], TASK_CLASSES,
        )
        calibration_fold_records.append({
            "fold": fold, "method": method,
            "temperature": fitted["temperature"],
            "calibration_windows": fitted["calibration_windows"],
            "calibration_nll_before": fitted["calibration_nll_before"],
            "calibration_nll_after": fitted["calibration_nll_after"],
            "train_subjects": len(train_people),
            "calibration_subjects": len(cal_people),
            "validation_subjects": len(valid_people),
            "validation_windows": len(validation_rows),
            "nll": multiclass_nll(
                y[validation_rows], probabilities, TASK_CLASSES
            ),
            "ece": metrics["ece_10_bins"],
            "brier": metrics["multiclass_brier"],
            "accuracy": metrics["accuracy"],
        })
        calibration_records.append(pd.DataFrame({
            "fold": fold, "method": method,
            "source_row": validation_rows,
            "subject_id": group_ids[validation_rows],
            "true_label": y[validation_rows],
            **{f"prob_{code}": probabilities[:, j]
               for j, code in enumerate(TASK_CLASSES)},
        }))

# -------------------- 3. Combine Outer Held-Out Evidence ---------------------
calibration_fold_results = pd.DataFrame(calibration_fold_records)
calibration_oof = pd.concat(calibration_records, ignore_index=True)
display(calibration_fold_results.round(4))


**09. Compare Held-Out Calibration Metrics by Fold**

Compare outer Validation NLL, ten-bin ECE, multiclass Brier and top-1 accuracy before and after scaling. Metrics on the Calibration partition are fitting diagnostics, not independent claims of improvement. The mean of three fold scores is distinguished from a pooled window-level metric.


In [ ]:
# ============= 09. Compare Held-Out Calibration Metrics by Fold ==============

# ---------------------- 1. Review Matched Fold Outcomes ----------------------
calibration_comparison = calibration_fold_results.pivot(
    index="fold", columns="method",
    values=["nll", "ece", "brier", "accuracy"],
)
display(calibration_comparison.round(4))
calibration_means = calibration_fold_results.groupby("method")[
    ["nll", "ece", "brier", "accuracy"]
].mean()
display(calibration_means.round(4))

# ------------------ 2. Plot Fold-Level Probability Metrics -------------------
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, metric in zip(axes, ["nll", "ece", "brier"]):
    for method, frame in calibration_fold_results.groupby(
        "method", sort=False
    ):
        ax.plot(frame["fold"], frame[metric], marker="o", label=method)
    ax.set(title=metric.upper(), xlabel="Outer Validation fold",
           xticks=list(range(1, FOLDS + 1)))
    ax.grid(alpha=0.2)
axes[0].legend()
fig.tight_layout()
save_figure(fig, output_dir / "08_calibration_fold_metrics.png")
plt.show()


**10. Inspect Out-of-Fold Reliability Before and After Scaling**

Compute both reliability curves from class-aligned predictions of the separately trained calibration experiment. The same outer held-out windows are evaluated in both conditions; calibration changes probability confidence but preserves the predicted class.


In [ ]:
# ======= 10. Inspect Out-of-Fold Reliability Before and After Scaling ========

# ------------------------ 1. Audit Paired Reliability ------------------------
calibration_reliability_tables = []
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
cal_prob_columns = [f"prob_{code}" for code in TASK_CLASSES]
for ax, (method, subset) in zip(
    axes, calibration_oof.groupby("method", sort=False)
):
    subset = subset.sort_values("source_row")
    probabilities = subset[cal_prob_columns].to_numpy()
    _, bins, _, metrics = audit_prediction_confidence(
        subset["true_label"].to_numpy(), probabilities,
        subset["subject_id"].to_numpy(), TASK_CLASSES,
    )
    filled = bins.loc[bins["windows"] > 0]
    calibration_reliability_tables.append(bins.assign(method=method))
    ax.plot([0, 1], [0, 1], "--", label="Ideal reference")
    ax.plot(
        filled["mean_confidence"], filled["observed_accuracy"],
        marker="o", label=method,
    )
    ax.set(
        xlim=(0, 1), ylim=(0, 1),
        title=f"{method} | ECE={metrics['ece_10_bins']:.4f}",
        xlabel="Mean confidence", ylabel="Observed accuracy",
    )
    ax.legend()
    ax.grid(alpha=0.2)
fig.tight_layout()
save_figure(fig, output_dir / "08_calibration_reliability.png")
plt.show()


**11. Verify and Export Disjoint-Subject Calibration Evidence**

Confirm that both probability conditions cover every original base source window exactly once per model, use identical labels and preserve top-1 predictions. Save participant-role assignments, fold and mean metrics, class-aligned probabilities and reliability tables without adding unneeded execution reports.


In [ ]:
# ======== 11. Verify and Export Disjoint-Subject Calibration Evidence ========

# ----------------- 1. Check Alignment and Top-1 Preservation -----------------
expected_rows = set(base_rows.tolist())
for method, table in calibration_oof.groupby("method"):
    if not table["source_row"].is_unique:
        raise ValueError(f"Duplicated held-out source rows for {method}.")
    if set(table["source_row"]) != expected_rows:
        raise ValueError(f"Unmatched outer-held-out rows for {method}.")
    if set(table["subject_id"]) & RESERVED_TEST:
        raise ValueError("Reserved Test participants entered calibration.")

uncalibrated = calibration_oof.loc[
    calibration_oof["method"] == "Uncalibrated"
].sort_values("source_row")
temperature_scaled = calibration_oof.loc[
    calibration_oof["method"] == "Temperature scaled"
].sort_values("source_row")
raw_scores = uncalibrated[cal_prob_columns].to_numpy()
scaled_scores = temperature_scaled[cal_prob_columns].to_numpy()
np.testing.assert_array_equal(
    uncalibrated["true_label"].to_numpy(),
    temperature_scaled["true_label"].to_numpy(),
)
np.testing.assert_array_equal(
    raw_scores.argmax(axis=1), scaled_scores.argmax(axis=1),
)
if not calibration_fold_results["temperature"].gt(0).all():
    raise ValueError("All fitted temperatures must be positive.")
if len(calibration_fold_results) != FOLDS * 2:
    raise ValueError("Calibration fold results are incomplete.")

# -------------------- 2. Save Calibration Research Tables --------------------
pd.DataFrame(calibration_subject_assignments).to_csv(
    output_dir / "08_calibration_fold_subject_assignments.csv",
    index=False,
)
calibration_fold_results.to_csv(
    output_dir / "08_calibration_fold_metrics.csv", index=False
)
calibration_means.rename_axis("method").reset_index().to_csv(
    output_dir / "08_calibration_mean_metrics.csv", index=False
)
calibration_oof.to_csv(
    output_dir / "08_calibration_oof_probabilities.csv", index=False
)
pd.concat(calibration_reliability_tables, ignore_index=True).to_csv(
    output_dir / "08_calibration_reliability_bins.csv", index=False
)
print("Saved separate subject-disjoint calibration evidence.")


**12. Package Stage 08 Research Outputs**

Produce one downloadable collection of current `08_confidence_` and `08_calibration_` evidence, retaining the distinct methods, fitted model populations, held-out source rows and scientific limitations of the two experiments.


In [ ]:
# =================== 12. Package Stage 08 Research Outputs ===================

# ------------------------ 1. Summarise Model Evidence ------------------------
print("Development participants:", len(DEVELOPMENT_SUBJECTS))
print("Base and expanded windows:", len(base_rows), len(X))
print("Confidence-audit folds:", len(confidence_fold_metrics))
print("Calibration fold/method records:", len(calibration_fold_results))
print("Separate 400-update CNN fits:", FOLDS * 2)

# ------------------------ 2. Package Current Outputs -------------------------
download_outputs_from_colab(
    folder="wearable_activity_recognition/output", stage="08"
)


**13. Scientific Interpretation and Research Boundaries**

**Separate model populations:** The two experiments share the same HARTH Development cohort and outer GroupKFold assignments but fit different CNN models. The confidence audit trains on all outer Training participants; the calibration study withholds an independent Calibration participant set from model training. Results from the uncalibrated audit and calibration experiment cannot be treated as paired conditions on the same fitted model.

**Uncalibrated softmax:** Accuracy, ten-bin ECE, multiclass Brier, class support and confident mistakes are descriptive held-out Development diagnostics. Softmax magnitude is not validated uncertainty; overlapping windows, unequal participant support and sparse Shuffling and stair activities limit inference.

**Temperature scaling:** A single positive scalar is fitted using labels from Calibration participants only and tested on the disjoint outer Validation group. Top-1 class predictions remain unchanged, so accuracy is invariant while NLL, ECE and Brier can move in different directions. Calibration-only fitting diagnostics are not unbiased estimates of improvements.

**Limits:** Both experiments are single-seed, three-fold Development evaluations; the models are not prospectively tested, and the reserved S023/S032 participants are never scored. These findings do not establish independent generalisation, clinical reliability or deployment-ready confidence thresholds.

**Results:** Add numerical interpretations after checking the integrated execution's `08_confidence_` and `08_calibration_` outputs, without importing historical results as newly measured evidence.
